In [2]:
import pandas as pd, re

df = pd.read_csv("20250513-20260311_main_df.csv")

def url_date_parts(u):
    u = str(u)
    m = re.search(r'/(20\d{2})/(\d{2})/(\d{2})/', u)      # /YYYY/MM/DD/
    if m: return pd.Timestamp(f"{m[1]}-{m[2]}-{m[3]}"), "day"
    m = re.search(r'/(20\d{2})/(\d{2})/', u)              # /YYYY/MM/
    if m: return pd.Timestamp(f"{m[1]}-{m[2]}-01"), "month"
    m = re.search(r'/(20\d{2})/', u)                      # /YYYY/
    if m: return pd.Timestamp(f"{m[1]}-01-01"), "year"
    return None, "none"

df['assigned'] = pd.to_datetime(df['date'], format='%Y%m%d', errors='coerce')
df[['url_date','precision']] = df['url'].apply(
    lambda u: pd.Series(url_date_parts(u)))

# How many URLs even contain a date?
print("URL date precision available:")
print(df['precision'].value_counts(), "\n")

# Day-level check (the strict, meaningful one)
day = df[df['precision'] == "day"].dropna(subset=['assigned'])
exact   = (day['assigned'].dt.date == day['url_date'].dt.date).mean()
within1 = (abs(day['assigned'] - day['url_date']).dt.days <= 1).mean()

print(f"Rows with a full URL date: {len(day)}")
print(f"  Exact day match:         {exact:.1%}")
print(f"  Within 1 day:            {within1:.1%}")
print(f"  Off by more than a day:  {1 - within1:.1%}")

URL date precision available:
precision
none     6077
day      1604
month     227
year      133
Name: count, dtype: int64 

Rows with a full URL date: 1604
  Exact day match:         71.9%
  Within 1 day:            92.9%
  Off by more than a day:  7.1%


In [3]:
print(f"Total rows: {len(df)}")
print(f"Unique URLs: {df['url'].nunique()}")
print(f"Duplicate rate: {1 - df['url'].nunique()/len(df):.1%}")

Total rows: 8041
Unique URLs: 8041
Duplicate rate: 0.0%
